In [1]:
import hashlib
import json
import time
import os

def sha256(data: str) -> str:
    return hashlib.sha256(data.encode("utf-8")).hexdigest()

In [2]:
class Block:
    def __init__(self, index: int, timestamp: float, data: dict, previous_hash: str, nonce: int = 0):
        self.index = index
        self.timestamp = timestamp
        self.data = data  # This will now contain file content and filename
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        # Deterministic serialization ensures if content changes, hash changes
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True, separators=(",", ":"))
        return sha256(block_string)

In [3]:
class Blockchain:
    def __init__(self, difficulty: int = 3):
        self.chain = []
        self.difficulty = difficulty
        self.create_genesis_block()

    def create_genesis_block(self):
        # Genesis block starts the chain with previous_hash "0"
        genesis = Block(index=0, timestamp=time.time(), data={"msg": "genesis"}, previous_hash="0")
        self.mine_block(genesis)
        self.chain.append(genesis)

    @property
    def last_block(self) -> Block:
        return self.chain[-1]

    def is_valid_proof(self, block: Block) -> bool:
        return block.hash.startswith("0" * self.difficulty)

    def mine_block(self, block: Block) -> Block:
        while True:
            block.hash = block.compute_hash()
            if self.is_valid_proof(block):
                return block
            block.nonce += 1

    def add_block(self, file_path: str) -> Block:
        # Read the actual content of the file
        with open(file_path, 'r') as f:
            content = f.read()
        
        # Store both filename and content in the block data
        data = {
            "filename": os.path.basename(file_path),
            "content": content
        }
        
        new_block = Block(
            index=self.last_block.index + 1,
            timestamp=time.time(),
            data=data,
            previous_hash=self.last_block.hash
        )
        self.mine_block(new_block)
        
        # Integrity check
        if new_block.previous_hash != self.last_block.hash or not self.is_valid_proof(new_block):
            raise ValueError("Invalid block—rejected.")
            
        self.chain.append(new_block)
        return new_block

    def is_chain_valid(self, curr_block=None, prev_block=None) -> bool:
        if curr_block and prev_block:
            if curr_block.previous_hash != prev_block.compute_hash():
                return False
            if curr_block.hash != curr_block.compute_hash() or not self.is_valid_proof(curr_block):
                return False
            return True

        # Default: Full chain verification from memory
        for i in range(1, len(self.chain)):
            curr = self.chain[i]
            prev = self.chain[i - 1]
            if curr.previous_hash != prev.hash or curr.compute_hash() != curr.hash:
                return False
            if not self.is_valid_proof(curr):
                return False
        return True

In [4]:
if __name__ == "__main__":
    bc = Blockchain(difficulty=4)
    
    # List of files to add to the blockchain
    files = ["a1.txt", "a2.txt", "a3.txt"]
    
    for file_name in files:
        if os.path.exists(file_name):
            print(f"Adding {file_name} to the blockchain...")
            bc.add_block(file_name)
        else:
            print(f"File {file_name} not found.")

    # Print results
    for b in bc.chain:
        print(f"Index: {b.index} | File: {b.data.get('filename', 'N/A')}")
        print(f"Content: {b.data.get('content', 'N/A')}")
        print(f"Hash: {b.hash}")
        print("-" * 60)

    print("Chain valid?", bc.is_chain_valid())

Adding a1.txt to the blockchain...
Adding a2.txt to the blockchain...
Adding a3.txt to the blockchain...
Index: 0 | File: N/A
Content: N/A
Hash: 000060d23a98c2fafca67971376e919bd6714d71c589cae9277dc8d9665eecf4
------------------------------------------------------------
Index: 1 | File: a1.txt
Content: hello.
Hash: 00007c95dd2aea63ec0511428aa5e26677d4f37133d6a6aed8be00b68c97e53f
------------------------------------------------------------
Index: 2 | File: a2.txt
Content: $10000 transaction done from A TO B..
Hash: 0000dbfe1a14f85ea1f6c0d457d9dc3959adc7919032ea390954ffcadc16d86e
------------------------------------------------------------
Index: 3 | File: a3.txt
Content: second
Hash: 0000965af1a6cb98796b22b426b26aa9b3d00205473cc28b0ccacf3eca03f576
------------------------------------------------------------
Chain valid? True


In [5]:
import os

def verify_chain_with_live_files(blockchain):
    print("--- 🛡️ Starting Live File Link Verification ---")
    is_overall_valid = True

    for i in range(1, len(blockchain.chain)):
        curr_block = blockchain.chain[i]
        prev_block = blockchain.chain[i-1]
        prev_filename = prev_block.data.get("filename")

        if prev_filename and os.path.exists(prev_filename):
            with open(prev_filename, 'r') as f:
                live_content = f.read()
            
            temp_data = {"filename": prev_filename, "content": live_content}
            temp_prev_block = Block(
                prev_block.index, 
                prev_block.timestamp, 
                temp_data, 
                prev_block.previous_hash, 
                prev_block.nonce
            )
            
            if not blockchain.is_chain_valid(curr_block, temp_prev_block):
                print(f"❌ TAMPER DETECTED: {prev_filename} has been modified!")
                is_overall_valid = False
                break
            else:
                print(f"✅ Link [{prev_filename} -> {curr_block.data.get('filename')}] is secure.")
        
        else:
            if i == 1 and prev_block.index == 0:
                print(f"✅ Genesis link is valid.")
            else:
                print(f"⚠️ Warning: Previous file '{prev_filename}' not found on disk.")
                is_overall_valid = False
                break

    if is_overall_valid:
        print("\n🏆 Result: Blockchain is perfectly synced with your live files.")
    else:
        print("\n🚨 Result: Verification FAILED. The files on your disk do not match the original blockchain")
    
    return is_overall_valid

verify_chain_with_live_files(bc)

--- 🛡️ Starting Live File Link Verification ---
✅ Genesis link is valid.
✅ Link [a1.txt -> a2.txt] is secure.
✅ Link [a2.txt -> a3.txt] is secure.

🏆 Result: Blockchain is perfectly synced with your live files.


True

In [6]:
def update_and_remine_chain(blockchain):
    print("--- 🔄 Updating Blockchain to match Live Files ---")
    
    # Skip Genesis (index 0) and start with the files
    for i in range(1, len(blockchain.chain)):
        block = blockchain.chain[i]
        filename = block.data.get("filename")
        
        if filename and os.path.exists(filename):
            # 1. Read the NEW data from the file
            with open(filename, 'r') as f:
                new_content = f.read()
            
            # 2. Update the block's data in memory
            block.data["content"] = new_content
            
            # 3. Fix the Link: Update previous_hash to match the actual previous block
            block.previous_hash = blockchain.chain[i-1].hash
            
            # 4. Re-mine: The data changed, so we MUST find a new valid hash
            print(f"Re-mining {filename}...")
            block.nonce = 0 # Reset nonce to start fresh mining
            blockchain.mine_block(block) 
            
    print("✅ Chain has been re-synchronized with live files.")

# Execute the update
update_and_remine_chain(bc)

--- 🔄 Updating Blockchain to match Live Files ---
Re-mining a1.txt...
Re-mining a2.txt...
Re-mining a3.txt...
✅ Chain has been re-synchronized with live files.


In [7]:
# Now when you run the verify function, it will pass
verify_chain_with_live_files(bc)

--- 🛡️ Starting Live File Link Verification ---
✅ Genesis link is valid.
✅ Link [a1.txt -> a2.txt] is secure.
✅ Link [a2.txt -> a3.txt] is secure.

🏆 Result: Blockchain is perfectly synced with your live files.


True

In [ ]:
    # Print results
for b in bc.chain:
    print(f"Index: {b.index} | File: {b.data.get('filename', 'N/A')}")
    print(f"Content: {b.data.get('content', 'N/A')}")
    print(f"Hash: {b.hash}")
    print("-" * 60)

Index: 0 | File: N/A
Content: N/A
Hash: 000060d23a98c2fafca67971376e919bd6714d71c589cae9277dc8d9665eecf4
------------------------------------------------------------
Index: 1 | File: a1.txt
Content: hello.
Hash: 00007c95dd2aea63ec0511428aa5e26677d4f37133d6a6aed8be00b68c97e53f
------------------------------------------------------------
Index: 2 | File: a2.txt
Content: $10000 transaction done from A TO B..
Hash: 0000dbfe1a14f85ea1f6c0d457d9dc3959adc7919032ea390954ffcadc16d86e
------------------------------------------------------------
Index: 3 | File: a3.txt
Content: second
Hash: 0000965af1a6cb98796b22b426b26aa9b3d00205473cc28b0ccacf3eca03f576
------------------------------------------------------------
